# Portfolio 3 — Multi-Agent Reinforcement Learning (Atari *Warlords*)

**Vak:** Autonomous Systems · De Haagse Hogeschool

Dit notebook is zowel het **rapport** als de **reproduceerbare pijplijn**. We
trainen een Deep RL-agent die in de multi-agent Atari-omgeving *Warlords*
concurreert met drie andere agenten, en vergelijken die met baselines.

> **Uitvoeren:** dit notebook draait op **Google Colab** (Linux + GPU). De
> Atari-backend `multi_agent_ale_py` heeft geen Windows-wheels; op Windows werkt
> alleen WSL2 of Colab. Kies in Colab *Runtime → Change runtime type → GPU*.

## 1. Inleiding & Probleemanalyse

### 1.1 De omgeving: *Warlords*
*Warlords* is een Atari-spel met **vier spelers** die elk een kasteel in een hoek
verdedigen met een paddle. Een bal stuitert rond; wordt je kasteel geraakt, dan
lig je eruit. De laatste speler die overblijft wint. In de PettingZoo-implementatie
(`warlords_v3`) betekent dit:

| Eigenschap | Waarde |
|---|---|
| Agenten | `first_0`, `second_0`, `third_0`, `fourth_0` (4) |
| Actieruimte | `Discrete(6)`: 0=noop, 1=fire, 2=up, 3=right, 4=left, 5=down |
| Observatie (`obs_type="ram"`) | 128 bytes console-RAM (`uint8`, 0–255) |
| Beloning | **spaarzaam/terminaal**: −1 als je kasteel valt, +1 als je als laatste overblijft |

Dit is een **competitieve, gemengde multi-agent-omgeving**: de agenten delen één
omgeving maar hebben tegengestelde belangen.

### 1.2 Waarom dit een multi-agent-probleem is
Vanuit het perspectief van één agent zijn de andere drie agenten onderdeel van de
omgeving. Omdat die anderen óók leren en hun gedrag veranderen, is de omgeving
**niet-stationair**: de optimale strategie verschuift terwijl tegenstanders beter
worden (Busoniu et al., 2008; Lowe et al., 2017). Een goede aanpak moet hier
expliciet rekening mee houden.

### 1.3 Een cruciale observatie over de RAM
In `warlords_v3` krijgen **alle vier de agenten dezelfde globale 128-byte RAM**
(zie `base_atari_env.py`: `{agent: obs for agent in self.agents}`). Er is dus geen
hoek-specifieke observatie en geen indicator van *welke* hoek je bent. Een enkel
gedeeld beleid kan de hoeken daardoor niet uit elkaar houden. Dit motiveert direct
onze keuze voor **independent learners**: vier afzonderlijke policies, één per hoek,
die elk hun eigen hoek leren verdedigen.

### 1.4 Keuze van algoritme en trainingsstrategie
- **Algoritme: PPO** (Proximal Policy Optimization; Schulman et al., 2017). PPO is
  een on-policy actor-critic methode die door zijn *clipped* objective stabiel en
  robuust traint met weinig hyperparameter-tuning. Dat is een groot voordeel in een
  niet-stationaire multi-agent-setting, waar waarde-gebaseerde methoden (zoals DQN)
  gevoeliger zijn voor instabiliteit (de Witt et al., 2020).
- **Trainingsstrategie: Independent PPO (IPPO) met generatie-gewijs bevroren
  tegenstanders.** Elke hoek heeft een eigen PPO-beleid. We temmen de
  niet-stationariteit door de tegenstanders per *generatie* te bevriezen: elk
  beleid traint tegen vaste snapshots van de andere drie, waarna alle snapshots
  geüpdatet worden. Dit is verwant aan *fictitious self-play* (Heinrich & Silver,
  2016) en aan IPPO (de Witt et al., 2020), dat verrassend sterk presteert in
  competitieve benchmarks.
- **Observatie: RAM.** Het toernooi gebruikt `obs_type="ram"`, en een kleine MLP op
  128 bytes traint veel sneller dan een CNN op pixels — ideaal voor de vele
  benodigde environment-stappen.

### 1.5 Baseline
Als referentie gebruiken we twee baselines (opdracht 2a): een **random policy** en
een **rule-based policy** die actief de eigen hoek patrouilleert en periodiek vuurt.

## 2. Setup (Colab)

De volgende cel installeert de libraries en downloadt de Atari-ROMs. **Herstart
daarna eenmalig de kernel** (Runtime → Restart) en ga verder vanaf paragraaf 2.2.

In [2]:
# 2.1 Installeer libraries (Colab). Sla over als alles al geïnstalleerd is.
!pip -q install "pettingzoo[atari]>=1.24" "autorom[accept-rom-license]" "stable-baselines3>=2.0.0" imageio imageio-ffmpeg tqdm rich
!AutoROM --accept-license -q
print("Klaar. Herstart nu zo nodig de kernel.")

  error: subprocess-exited-with-error
  
  × Building wheel for multi_agent_ale_py (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [220 lines of output]
      running bdist_wheel
      running build
      running build_py
      creating build\lib.win-amd64-cpython-313\multi_agent_ale_py
      copying multi_agent_ale_py\ale_python_interface.py -> build\lib.win-amd64-cpython-313\multi_agent_ale_py
      copying multi_agent_ale_py\__init__.py -> build\lib.win-amd64-cpython-313\multi_agent_ale_py
      running egg_info
      writing multi_agent_ale_py.egg-info\PKG-INFO
      writing dependency_links to multi_agent_ale_py.egg-info\dependency_links.txt
      writing requirements to multi_agent_ale_py.egg-info\requires.txt
      writing top-level names to multi_agent_ale_py.egg-info\top_level.txt
      reading manifest file 'multi_agent_ale_py.egg-info\SOURCES.txt'
      reading manifest template 'MANIFEST.in'
      adding license file 'LICENSE.md'
      writing manifest fil

Klaar. Herstart nu zo nodig de kernel.


'AutoROM' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
# 2.2 Haal de projectcode op en zet het werkpad op Portfolio3/.
#     Let op: de Portfolio 3-code staat op de branch 'portfolio3_michal', dus we
#     klonen expliciet die branch (-b). Werkt het clonen niet? Upload dan de map
#     Portfolio3 handmatig naar Colab (mapicoon links) en draai deze cel opnieuw.
import os, sys

if not os.path.isdir("adsai-autonomous-systems"):
    !git clone -q -b portfolio3_michal https://github.com/HenryLau08/adsai-autonomous-systems.git

# Zoek de map die het pakket 'warlords_marl' bevat (na clone of na upload).
for cand in ["adsai-autonomous-systems/Portfolio3", "Portfolio3", "."]:
    if os.path.isdir(os.path.join(cand, "warlords_marl")):
        os.chdir(cand)
        break
sys.path.insert(0, os.getcwd())

assert os.path.isdir("warlords_marl"), (
    "Map 'warlords_marl' niet gevonden. Push je code eerst naar de branch "
    "'portfolio3_michal', of upload de map Portfolio3 handmatig naar Colab."
)
print("Werkmap:", os.getcwd())

Werkmap: c:\Users\alr3m\Documents\GitHub\adsai-autonomous-systems\Portfolio3


In [6]:
# 2.3 Imports en reproduceerbaarheid.
%pip install matplotlib
import numpy as np
import matplotlib.pyplot as plt

from warlords_marl import AGENT_ORDER, ACTION_MEANINGS
from warlords_marl.env_wrapper import make_parallel_env
from warlords_marl.baselines import RandomPolicy, RuleBasedPolicy, PolicyOpponent
from warlords_marl import ram_tools, train, evaluate

SEED = 0
np.random.seed(SEED)
print("Agenten:", AGENT_ORDER)
print("Acties:", ACTION_MEANINGS)

Note: you may need to restart the kernel to use updated packages.


error: externally-managed-environment

× This environment is externally managed
╰─> This Python installation is managed by uv and should not be modified.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detailed specification.


ModuleNotFoundError: No module named 'matplotlib'

## 3. De omgeving verkennen

We controleren de actie- en observatieruimte en bekijken welke RAM-bytes het meest
veranderen — die zijn kandidaat voor bal- en paddleposities (de starterscode bevat
hiervoor de basis; `ram_tools` aggregeert dit over een hele episode).

In [ ]:
# 3.1 Actie- en observatieruimte controleren.
env = make_parallel_env(obs_type="ram")
obs, _ = env.reset(seed=SEED)
a0 = env.agents[0]
print("Aantal agenten:", len(env.agents))
print("Actieruimte:", env.action_space(a0))
print("Observatieruimte:", env.observation_space(a0))
print("RAM shape:", np.asarray(obs[a0]).shape, "dtype:", np.asarray(obs[a0]).dtype)
print("Alle agenten zien dezelfde RAM:",
      all(np.array_equal(obs[a0], obs[a]) for a in env.agents))
env.close()

In [ ]:
# 3.2 Welke RAM-bytes veranderen het meest? (kandidaten voor bal/paddle)
env = make_parallel_env(obs_type="ram")
policies = {a: RandomPolicy(seed=SEED) for a in AGENT_ORDER}
trace = ram_tools.collect_ram_trace(env, policies, max_steps=1500, seed=SEED)
print("Trace shape:", trace.shape)
print("\nTop-12 meest veranderende bytes (index, #unieke waarden, std):")
for idx, uniq, std in ram_tools.rank_changing_bytes(trace, top_k=12):
    print(f"  byte {idx:3d}: {uniq:3d} unieke waarden, std={std:5.1f}")
# Vul deze indexen later in bij RuleBasedPolicy(ball_byte=..., paddle_byte=...)

## 4. Baselines

We meten eerst hoe random- en rule-based policies presteren. Dit is het
referentiepunt waartegen we het RL-beleid afzetten (opdracht 3b).

In [ ]:
# 4.1 Baseline-toernooi: 4x random vs. 4x rule-based.
random_policies = {a: RandomPolicy(seed=SEED) for a in AGENT_ORDER}
rule_policies = {a: RuleBasedPolicy(corner=a) for a in AGENT_ORDER}

print("== 4x random ==")
summary_random = evaluate.run_tournament(make_parallel_env, random_policies,
                                         n_games=20, verbose=False)
print(summary_random["win_rate"], "\n")

print("== 4x rule-based ==")
summary_rule = evaluate.run_tournament(make_parallel_env, rule_policies,
                                       n_games=20, verbose=False)
print(summary_rule["win_rate"])

## 5. Methode: van multi-agent naar trainbaar single-agent

Stable-Baselines3 traint één beleid in een standaard Gymnasium-omgeving. Onze
`WarlordsSingleAgentEnv` (in `warlords_marl/env_wrapper.py`) overbrugt dit: SB3
bestuurt precies één hoek, terwijl de andere drie hoeken door *opponent*-policies
worden gespeeld. Belangrijke ontwerpkeuzes:

- **Observatie-normalisatie:** ruwe RAM-bytes (0–255) worden geschaald naar
  `[0, 1]`, wat het leren met een MLP stabiliseert.
- **Reward shaping:** de ruwe beloning is spaarzaam (alleen ±1 op het eind). We
  voegen een kleine `survival_bonus` per overleefde stap toe, zodat "langer
  overleven" beloond wordt — dat correleert sterk met winnen en versnelt het leren
  (vgl. reward shaping, Ng et al., 1999). De *evaluatie* gebruikt altijd de pure
  win/verlies-uitkomst, niet de shaping.
- **Independent learners:** elke hoek krijgt een eigen PPO-model; tegenstanders
  worden per generatie bevroren (`train.train_independent`).

De boekhouding van deze wrapper (actie-dict, doorgeven van beloning/terminatie,
afhandelen van geëlimineerde agenten) is lokaal geverifieerd met unit-tests
(`tests/test_env_wrapper.py`).

## 6. Training (IPPO)

> **Tijd vs. kwaliteit.** Hieronder staan bescheiden waarden voor een snelle
> demonstratie. Voor sterke agenten verhoog je `steps_per_agent` (bijv. 300k–500k)
> en `generations` (3–5). Op een Colab-GPU met RAM-observaties is de emulator
> (CPU) de bottleneck, dus meer `n_envs` (parallelle omgevingen) helpt het meest.

In [ ]:
# 6.1 Train vier independent PPO-policies (één per hoek).
#     Demo-instelling; schaal op voor betere resultaten.
models = train.train_independent(
    generations=2,
    steps_per_agent=50_000,
    n_envs=8,
    survival_bonus=0.01,
    seed=SEED,
    save_dir="tournament/models",   # Agent3/Agent4 lezen hieruit
    device="auto",
    progress_bar=True,
)
print("Getrainde hoeken:", list(models.keys()))

## 7. Resultaten & Discussie

### 7.1 Leercurves
De trainingsbeloning per hoek (geglad over episodes), samengevoegd over generaties.
Een stijgende curve duidt op leren; let ook op de **stabiliteit** (ruis/variantie).

In [ ]:
# 7.1 Leercurves uit de monitor-logs.
import os
fig = evaluate.plot_learning_curves(
    monitor_root="tournament/models/monitor",
    window=50,
    save_path="attachments/leercurves.png",
)
plt.show()

### 7.2 RL vs. baseline
We laten de getrainde policies een toernooi spelen en vergelijken het
win-percentage met de baselines. Ter controle laten we ook één PPO-hoek tegen drie
random-tegenstanders spelen: een eerlijke "leert RL daadwerkelijk iets?"-test.

In [ ]:
# 7.2 Bouw policies uit de getrainde modellen en evalueer.
from stable_baselines3 import PPO

ppo_models = {a: PPO.load(f"tournament/models/ppo_{a}.zip", device="cpu")
              for a in AGENT_ORDER}
ppo_policies = {a: PolicyOpponent(ppo_models[a], deterministic=True)
                for a in AGENT_ORDER}

# 4x PPO (independent learners tegen elkaar)
summary_ppo = evaluate.run_tournament(make_parallel_env, ppo_policies,
                                      n_games=20, verbose=False)

# 1x PPO (first_0) vs 3x random -> isoleert het effect van RL
mixed = {a: ppo_policies[a] if a == "first_0" else RandomPolicy(seed=SEED)
         for a in AGENT_ORDER}
summary_mixed = evaluate.run_tournament(make_parallel_env, mixed,
                                        n_games=20, verbose=False)

print("PPO first_0 win-rate vs 3x random:", summary_mixed["win_rate"]["first_0"])

In [ ]:
# 7.3 Vergelijk win-percentages over configuraties.
fig = evaluate.plot_winrates(
    {
        "4x random": summary_random,
        "4x rule-based": summary_rule,
        "4x PPO": summary_ppo,
        "PPO vs 3x random": summary_mixed,
    },
    save_path="attachments/winrates.png",
)
plt.show()

### 7.4 Een wedstrijd opnemen (optioneel)
Leg een wedstrijd vast als video om het gedrag kwalitatief te beoordelen.

In [ ]:
# 7.4 Neem één PPO-wedstrijd op als mp4.
import imageio
env = make_parallel_env(obs_type="ram", render_mode="rgb_array")
result = evaluate.play_match(env, ppo_policies, seed=123, record_frames=True)
os.makedirs("warlords_videos", exist_ok=True)
imageio.mimsave("warlords_videos/ppo_match.mp4", result["frames"], fps=15)
print("Winnaar:", result["winner"], "| stappen:", result["steps"])

## 8. Hyperparameter-experimenten (opdracht 3a)

We variëren één hyperparameter tegelijk en houden de rest vast, om het effect te
isoleren. Hieronder een sjabloon voor de **learning rate**; herhaal analoog voor
bijvoorbeeld `ent_coef` (exploratie), `survival_bonus` (reward shaping) of de
netwerkgrootte. Documenteer telkens de win-rate en de stabiliteit van de leercurve.

> Houd het aantal stappen klein per run als je meerdere waarden vergelijkt, of
> draai deze sectie als aparte (langere) experimenten en vat de resultaten samen
> in een tabel.

In [ ]:
# 8.1 Sjabloon: vergelijk learning rates (klein gehouden voor snelheid).
results_lr = {}
for lr in [1e-4, 2.5e-4, 5e-4]:
    tag = f"lr={lr}"
    print(f"\n=== {tag} ===")
    train.train_independent(
        generations=1, steps_per_agent=20_000, n_envs=8,
        survival_bonus=0.01, seed=SEED,
        save_dir=f"experiments/{tag}",
        progress_bar=False,
        ppo_overrides=dict(learning_rate=lr),
    )
    m = {a: PPO.load(f"experiments/{tag}/ppo_{a}.zip", device="cpu") for a in AGENT_ORDER}
    pol = {a: PolicyOpponent(m[a], deterministic=True) for a in AGENT_ORDER}
    mixed = {a: pol[a] if a == "first_0" else RandomPolicy(seed=SEED) for a in AGENT_ORDER}
    s = evaluate.run_tournament(make_parallel_env, mixed, n_games=15, verbose=False)
    results_lr[tag] = s["win_rate"]["first_0"]

print("\nWin-rate (PPO first_0 vs 3x random) per learning rate:")
for tag, wr in results_lr.items():
    print(f"  {tag}: {wr:.2f}")

## 9. Toernooi-agent (bonus)

Voor het klassentoernooi wordt onze agent in een **willekeurige** hoek geplaatst en
krijgt hij alleen de globale RAM (geen hoek-indicator). Eén beleid kan dan niet voor
elke hoek tegelijk optimaal zijn (zie §1.3). We trainen daarom een **hoek-robuust**
beleid dat afwisselend alle hoeken bestuurt; `Agent3`/`Agent4` vallen hierop terug
als er geen hoek-specifiek model is.

In [ ]:
# 9.1 (optioneel) Train een hoek-robuust beleid voor het toernooi.
robust = train.train_corner_robust(
    total_timesteps=100_000, n_envs=8, survival_bonus=0.01,
    seed=SEED, save_dir="tournament/models", progress_bar=True,
)
print("Hoek-robuust model opgeslagen als tournament/models/ppo_corner_robust.zip")

Draai vervolgens `tournament/warlords_tournament_ram_mode.ipynb` (met de vier
`agentN.py`-bestanden) om Agent1 (random) en Agent2 (rule-based) tegen Agent3/Agent4
(PPO) te laten spelen — een directe demonstratie van de meerwaarde van RL.

## 10. Conclusie & Reflectie

**Samenvatting.** We hebben een MARL-systeem voor *Warlords* gebouwd: vier
independent PPO-policies, getraind met generatie-gewijs bevroren tegenstanders,
bovenop een single-agent wrapper rond de PettingZoo parallel-omgeving. De
prestaties zijn afgezet tegen een random- en een rule-based baseline.

**Wat biedt RL hier?** *(Vul in met je eigen resultaten.)* In tegenstelling tot de
vaste baselines past het RL-beleid zich aan de bal- en tegenstanderdynamiek aan en
verbetert het meetbaar gedurende de training (zie de leercurves en het hogere
win-percentage in §7).

**Beperkingen.**
- De beloning is zeer spaarzaam; zonder de survival-shaping leert PPO traag.
- Omdat alle hoeken dezelfde globale RAM zien zonder hoek-indicator, kan één
  gedeeld beleid de hoeken niet onderscheiden — dit beperkt een hoek-agnostische
  toernooi-agent fundamenteel.
- Generatie-gewijs bevriezen benadert echte gelijktijdige independent learning, maar
  is niet identiek (vgl. de Witt et al., 2020).

**Mogelijke uitbreidingen.**
- Een hoek-indicator toevoegen aan de observatie (bijv. one-hot), zodat één gedeeld
  beleid wél per hoek kan specialiseren (parameter sharing met agent-ID).
- Self-play met een *pool* van eerdere snapshots (PSRO / league-training) i.p.v. één
  generatie, voor robuustere strategieën.
- RAM-bytes kalibreren (§3.2) en in de reward gebruiken (bijv. balcontrole belonen).
- Vergelijken met een waarde-gebaseerde methode (DQN) of met pixel-observaties +
  CNN.

## 11. Verantwoording Generative AI

> Volgens de cursusregels moet elk gebruik van Generative AI (GenAI) worden
> verantwoord met promptnummer, titel en link, plus een niet-GenAI-bron die de
> juistheid bevestigt. Vul hieronder je eigen prompts in (verwijder dit blok niet).

**Voorbeeld (format):**
- [ChatGPT, 2026. Prompt 1: PPO voor multi-agent Warlords](https://chat.openai.com/share/…) —
  bevestigd door Schulman et al. (2017) en Raffin et al. (2021).

In code-cellen verwijs je in een comment naar de prompt, bijv.:
`# Prompt 1: PPO voor multi-agent Warlords`.

## 12. Referentielijst (APA)

- Bellemare, M. G., Naddaf, Y., Veness, J., & Bowling, M. (2013). The Arcade
  Learning Environment: An evaluation platform for general agents. *Journal of
  Artificial Intelligence Research, 47*, 253–279.
- Busoniu, L., Babuška, R., & De Schutter, B. (2008). A comprehensive survey of
  multiagent reinforcement learning. *IEEE Transactions on Systems, Man, and
  Cybernetics, 38*(2), 156–172.
- de Witt, C. S., Gupta, T., Makoviichuk, D., Makoviychuk, V., Torr, P. H. S.,
  Sun, M., & Whiteson, S. (2020). *Is independent learning all you need in the
  StarCraft multi-agent challenge?* arXiv:2011.09533.
- Heinrich, J., & Silver, D. (2016). *Deep reinforcement learning from self-play in
  imperfect-information games.* arXiv:1603.01121.
- Lowe, R., Wu, Y., Tamar, A., Harb, J., Abbeel, P., & Mordatch, I. (2017).
  Multi-agent actor-critic for mixed cooperative-competitive environments.
  *Advances in Neural Information Processing Systems, 30*.
- Mnih, V., Kavukcuoglu, K., Silver, D., et al. (2015). Human-level control through
  deep reinforcement learning. *Nature, 518*(7540), 529–533.
- Ng, A. Y., Harada, D., & Russell, S. (1999). Policy invariance under reward
  transformations: Theory and application to reward shaping. *ICML*, 278–287.
- Raffin, A., Hill, A., Gleave, A., Kanervisto, A., Ernestus, M., & Dormann, N.
  (2021). Stable-Baselines3: Reliable reinforcement learning implementations.
  *Journal of Machine Learning Research, 22*(268), 1–8.
- Schulman, J., Wolski, F., Dhariwal, P., Radford, A., & Klimov, O. (2017).
  *Proximal policy optimization algorithms.* arXiv:1707.06347.
- Sutton, R. S., & Barto, A. G. (2018). *Reinforcement learning: An introduction*
  (2nd ed.). MIT Press.
- Terry, J. K., Black, B., Grammel, N., et al. (2021). PettingZoo: Gym for
  multi-agent reinforcement learning. *Advances in Neural Information Processing
  Systems, 34*.